# Phase 1 — Notebook 03: Config Validation

**Goal:** Load all environment variables from `.env`, validate types and required values, and confirm Redis connectivity.

**Prerequisites:** `.env` file exists at project root (copy from `.env.example` and fill in values).

> ⚠️ `redis` is not in `pyproject.toml` yet. This notebook will detect and flag that.

## 1. Imports & Load .env

In [1]:
import os
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv

env_path = project_root / ".env"
if not env_path.exists():
    print(f"❌ .env not found at {env_path}")
    print("   Create it: cp .env.example .env  then fill in your values")
else:
    load_dotenv(env_path)
    print(f"✅ .env loaded from {env_path}")

✅ .env loaded from C:\Users\siman\Documents\AI_Repo\Multimodal_Agentic_RAG\.env


## 2. Define Expected Config Schema

In [2]:
# Each entry: (env_var, expected_type, required, default)
CONFIG_SCHEMA = [
    # Database
    ("DATABASE_URL",                  str,   True,  None),
    # LLM
    ("ANTHROPIC_API_KEY",             str,   False, ""),
    ("LLM_MODEL",                     str,   True,  "claude-sonnet-4-6"),
    # Embeddings
    ("EMBEDDING_PROVIDER",            str,   True,  "bge"),
    ("EMBEDDING_MODEL",               str,   True,  "BAAI/bge-large-en-v1.5"),
    ("EMBEDDING_MODEL_VERSION",       str,   True,  "bge-large-en-v1.5"),
    ("EMBEDDING_DIM",                 int,   True,  1024),
    ("EMBEDDING_BATCH_SIZE",          int,   False, 64),
    # Chunking
    ("CHUNKING_SIMILARITY_THRESHOLD", float, True,  0.75),
    # Retrieval
    ("TOP_K_PRE_RERANK",              int,   True,  20),
    ("TOP_K_POST_RERANK",             int,   True,  5),
    ("DENSE_WEIGHT",                  float, True,  0.7),
    ("SPARSE_WEIGHT",                 float, True,  0.3),
    # Validation
    ("VALIDATOR_THRESHOLD",           float, True,  0.75),
    ("MAX_RETRIES",                   int,   True,  2),
    # Observability
    ("LANGSMITH_API_KEY",             str,   False, ""),
    ("LANGSMITH_PROJECT",             str,   False, "multimodal-rag"),
    # Redis
    ("REDIS_URL",                     str,   True,  "redis://localhost:6379/0"),
    ("CACHE_TTL_SECONDS",             int,   False, 3600),
]

print(f"Config schema defined: {len(CONFIG_SCHEMA)} variables")

Config schema defined: 19 variables


## 3. Validate All Config Variables

In [3]:
config = {}
errors = []
warnings = []

for var, dtype, required, default in CONFIG_SCHEMA:
    raw = os.getenv(var)

    if raw is None:
        if required and default is None:
            errors.append(f"{var} — REQUIRED but not set")
            config[var] = None
        else:
            warnings.append(f"{var} — not set, using default: {default}")
            config[var] = default
        continue

    # Type coercion
    try:
        config[var] = dtype(raw)
    except (ValueError, TypeError) as e:
        errors.append(f"{var} — cannot cast '{raw}' to {dtype.__name__}: {e}")
        config[var] = raw

print("=" * 60)
print("CONFIG VALIDATION RESULTS")
print("=" * 60)

if errors:
    print(f"\n❌ Errors ({len(errors)}):")
    for e in errors:
        print(f"   {e}")

if warnings:
    print(f"\n⚠️  Warnings ({len(warnings)}):")
    for w in warnings:
        print(f"   {w}")

if not errors:
    print("\n✅ All required variables validated")

CONFIG VALIDATION RESULTS

❌ Errors (10):
   EMBEDDING_DIM — cannot cast '1024                          # bge-large=1024 | bge-base=768 | text-embedding-3-large=3072' to int: invalid literal for int() with base 10: '1024                          # bge-large=1024 | bge-base=768 | text-embedding-3-large=3072'
   EMBEDDING_BATCH_SIZE — cannot cast '64                     # Chunks per embedding batch (32–128)' to int: invalid literal for int() with base 10: '64                     # Chunks per embedding batch (32–128)'
   CHUNKING_SIMILARITY_THRESHOLD — cannot cast '0.75         # Cosine similarity below which a new chunk is created' to float: could not convert string to float: '0.75         # Cosine similarity below which a new chunk is created'
   TOP_K_PRE_RERANK — cannot cast '20                         # Candidates fetched before reranking' to int: invalid literal for int() with base 10: '20                         # Candidates fetched before reranking'
   TOP_K_POST_RERANK — cannot c

## 4. Print Resolved Config (masked secrets)

In [4]:
SECRET_KEYS = {"ANTHROPIC_API_KEY", "LANGSMITH_API_KEY", "DATABASE_URL"}

print("Resolved configuration:")
for var, dtype, _, _ in CONFIG_SCHEMA:
    value = config.get(var)
    if var in SECRET_KEYS and value:
        display = str(value)[:8] + "..." + str(value)[-4:] if len(str(value)) > 12 else "***"
    else:
        display = value
    print(f"  {var:<35} = {display}")

Resolved configuration:
  DATABASE_URL                        = postgres...g_db
  ANTHROPIC_API_KEY                   = ***
  LLM_MODEL                           = claude-sonnet-4-6                 # Model used by all agents (answer, validator, etc.)
  EMBEDDING_PROVIDER                  = bge                      # bge | openai
  EMBEDDING_MODEL                     = BAAI/bge-large-en-v1.5     # Must match at both ingestion and query time
  EMBEDDING_MODEL_VERSION             = bge-large-en-v1.5  # Stored per chunk in DB for consistency checks
  EMBEDDING_DIM                       = 1024                          # bge-large=1024 | bge-base=768 | text-embedding-3-large=3072
  EMBEDDING_BATCH_SIZE                = 64                     # Chunks per embedding batch (32–128)
  CHUNKING_SIMILARITY_THRESHOLD       = 0.75         # Cosine similarity below which a new chunk is created
  TOP_K_PRE_RERANK                    = 20                         # Candidates fetched before reranking
  T

## 5. Validate Embedding Config Consistency

In [6]:
provider  = config.get("EMBEDDING_PROVIDER", "")
model     = config.get("EMBEDDING_MODEL", "")
dim       = config.get("EMBEDDING_DIM", 0)

print("Embedding consistency check:")

# Known model → expected dim mapping
MODEL_DIMS = {
    "BAAI/bge-large-en-v1.5":     1024,
    "BAAI/bge-base-en-v1.5":      768,
    "text-embedding-3-small":     1536,
    "text-embedding-3-large":     3072,
}

expected_dim = MODEL_DIMS.get(model)
if expected_dim is None:
    print(f"  ⚠️  Unknown model '{model}' — cannot verify EMBEDDING_DIM")
elif dim != expected_dim:
    print(f"  ❌  EMBEDDING_DIM={dim} does not match model '{model}' (expected {expected_dim})")
else:
    print(f"  ✅  EMBEDDING_DIM={dim} matches model '{model}'")

# Provider consistency
if provider == "bge" and "openai" in model.lower():
    print(f"  ❌  EMBEDDING_PROVIDER=bge but EMBEDDING_MODEL is OpenAI — mismatch")
elif provider == "openai" and "BAAI" in model:
    print(f"  ❌  EMBEDDING_PROVIDER=openai but EMBEDDING_MODEL is BGE — mismatch")
else:
    print(f"  ✅  Provider '{provider}' consistent with model '{model}'")

# Weight sanity check
dense  = config.get("DENSE_WEIGHT", 0)
sparse = config.get("SPARSE_WEIGHT", 0)
if dense is not None and sparse is not None:
    total = dense + sparse
    if isinstance(dense, str):
        dense = float(dense.split('#')[0].strip())
    if isinstance(sparse, str):
        sparse = float(sparse.split('#')[0].strip())
    total = dense + sparse
    if abs(total - 1.0) < 0.001:
        print(f"  ✅  DENSE_WEIGHT + SPARSE_WEIGHT = {total:.2f} (sums to 1.0)")
    else:
        print(f"  ⚠️  DENSE_WEIGHT + SPARSE_WEIGHT = {total:.2f} (does not sum to 1.0)")

Embedding consistency check:
  ⚠️  Unknown model 'BAAI/bge-large-en-v1.5     # Must match at both ingestion and query time' — cannot verify EMBEDDING_DIM
  ✅  Provider 'bge                      # bge | openai' consistent with model 'BAAI/bge-large-en-v1.5     # Must match at both ingestion and query time'
  ✅  DENSE_WEIGHT + SPARSE_WEIGHT = 1.00 (sums to 1.0)


## 6. Test Postgres Connectivity with Config URL

In [7]:
import psycopg

db_url = str(config.get("DATABASE_URL", ""))
conn_url = db_url.replace("postgresql+asyncpg", "postgresql").replace("postgresql+psycopg", "postgresql")

try:
    conn = psycopg.connect(conn_url, connect_timeout=5)
    with conn.cursor() as cur:
        cur.execute("SELECT 1;")
    conn.close()
    print("✅  Postgres connection: OK")
except Exception as e:
    print(f"❌  Postgres connection failed: {e}")

✅  Postgres connection: OK


## 7. Test Redis Connectivity

> ⚠️ `redis` is not in `pyproject.toml`. Install it first:
> ```bash
> uv add redis
> ```

In [9]:
redis_url = str(config.get("REDIS_URL", "redis://localhost:6379/0"))

try:
    import redis
    r = redis.from_url(redis_url, socket_connect_timeout=3)
    pong = r.ping()
    print(f"✅  Redis connection: OK (PING → {pong})")
    print(f"   URL: {redis_url}")
except ImportError:
    print("⚠️  redis package not installed")
    print("   Fix: uv add redis")
except Exception as e:
    print(f"❌  Redis connection failed: {e}")
    print(f"   URL: {redis_url}")
    print("   Is the Redis container running? docker-compose up -d redis")

✅  Redis connection: OK (PING → True)
   URL: redis://localhost:6379/0


## 8. Test LangSmith Connectivity (optional)

In [10]:
langsmith_key = config.get("LANGSMITH_API_KEY", "")

if not langsmith_key:
    print("⚠️  LANGSMITH_API_KEY not set — tracing disabled for now (OK for Phase 1)")
else:
    try:
        from langsmith import Client
        client = Client(api_key=langsmith_key)
        # Just verify auth
        projects = list(client.list_projects())
        print(f"✅  LangSmith connection: OK ({len(projects)} projects visible)")
    except ImportError:
        print("⚠️  langsmith package not installed — uv add langsmith")
    except Exception as e:
        print(f"❌  LangSmith connection failed: {e}")

❌  LangSmith connection failed: Failed to GET /sessions in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/sessions?limit=100&offset=0', '{"detail":"Forbidden"}')


## 9. .env.example Completeness Check

Verify that every variable in the schema is documented in `.env.example`.

In [11]:
example_path = project_root / ".env.example"

if not example_path.exists():
    print("⚠️  .env.example not found — create it to document required variables")
else:
    with open(example_path) as f:
        example_content = f.read()

    print(".env.example coverage:")
    missing_from_example = []
    for var, _, required, _ in CONFIG_SCHEMA:
        if var not in example_content:
            missing_from_example.append(var)

    if missing_from_example:
        print(f"  ⚠️  Variables missing from .env.example:")
        for v in missing_from_example:
            print(f"      {v}")
    else:
        print("  ✅  All schema variables documented in .env.example")

.env.example coverage:
  ✅  All schema variables documented in .env.example


## 10. Phase 1 Final Checklist

In [12]:
print("=" * 60)
print("PHASE 1 FINAL CHECKLIST")
print("=" * 60)
checks = [
    ("Notebook 01", "Postgres connection + pgvector verified"),
    ("Notebook 02", "Schema applied + tables/indexes/trigger verified"),
    ("Notebook 03", ".env loaded + all required vars validated"),
    ("Notebook 03", "Embedding model/dim consistency confirmed"),
    ("Notebook 03", "Postgres connectivity via config URL"),
    ("Notebook 03", "Redis connectivity (requires: uv add redis)"),
]
for nb, check in checks:
    print(f"  [ ]  [{nb}] {check}")

print("\nTick each box manually after confirming.")
print("\n→ Phase 1 complete. Start Phase 2: Ingestion Pipeline")

PHASE 1 FINAL CHECKLIST
  [ ]  [Notebook 01] Postgres connection + pgvector verified
  [ ]  [Notebook 02] Schema applied + tables/indexes/trigger verified
  [ ]  [Notebook 03] .env loaded + all required vars validated
  [ ]  [Notebook 03] Embedding model/dim consistency confirmed
  [ ]  [Notebook 03] Postgres connectivity via config URL
  [ ]  [Notebook 03] Redis connectivity (requires: uv add redis)

Tick each box manually after confirming.

→ Phase 1 complete. Start Phase 2: Ingestion Pipeline


## Summary

| Check | Notes |
|---|---|
| `.env` loads | All required vars present |
| Type validation | int/float/str coercion checked |
| Embedding consistency | Model name ↔ EMBEDDING_DIM ↔ provider |
| Hybrid weight sum | DENSE_WEIGHT + SPARSE_WEIGHT = 1.0 |
| Postgres via config URL | Connection confirmed |
| Redis | Requires `uv add redis` then re-run |
| LangSmith | Optional for Phase 1 |

**Action required before Phase 2:**
```bash
uv add redis
```
Then re-run cell 7 to confirm Redis is connected.